# DACN - Train EMBER2024 (Win64) trên Kaggle → 1 file zip
**Trước khi chạy:** Settings → Internet **On**; Accelerator **None**.
Nên bấm **Save Version → Save & Run All (Commit)** để chạy nền; kết quả ở tab **Output**: `dacn_ml_bundle_win64.zip`.
Bị kill vì hết RAM thì tự thử lại với mẫu 400.000 dòng.
Chỉ phản ánh EMBER2024, **không phải** chất lượng phát hiện của DACN.

In [ ]:
# =====================================================================
# DACN - Train Win64 trên EMBER2024, gom kết quả thành 1 file zip.
# Mỗi loại chạy trong tiến trình riêng (giải phóng RAM giữa các lần).
# Loại nào đã có báo cáo trong /kaggle/working thì bỏ qua. Một loại lỗi không làm mất loại kia.
# Kết quả chỉ phản ánh EMBER2024, KHÔNG phải chất lượng phát hiện của DACN.
# =====================================================================
import os, sys, io, json, time, base64, gzip, shutil, hashlib, zipfile, subprocess

TYPES = os.environ.get("DACN_TYPES", "Win64").split(",")
WORK  = os.environ.get("DACN_WORK", "/kaggle/working")
FALLBACK_ROWS = 400_000          # nếu lần đầu bị kill (hết RAM), thử lại với số dòng này
KEEP_DATA = os.environ.get("DACN_KEEP_DATA") == "1"   # True = giữ file .dat sau khi xong (tốn đĩa)
T0 = time.time()
os.makedirs(WORK, exist_ok=True)

SCRIPT_B64 = "..."
script_path = "/tmp/train_ember2024.py"
open(script_path, "wb").write(gzip.decompress(base64.b64decode(SCRIPT_B64)))

def log(m): print(f"[{(time.time()-T0)/60:7.1f} phút] {m}", flush=True)

def run_type(loai):
    ten = loai.lower().replace("_", "")
    rep_path = f"{WORK}/report_{ten}.json"
    if os.path.exists(rep_path):
        log(f"{loai}: đã có báo cáo, bỏ qua"); return "SKIPPED_DONE"
    data = os.environ.get("DACN_DATA") or f"/tmp/data/{ten}"
    for attempt, rows in enumerate((None, FALLBACK_ROWS), start=1):
        env = dict(os.environ, DACN_LOAI=loai, DACN_WORK=WORK, DACN_DATA=data)
        if rows: env["DACN_MAX_ROWS"] = str(rows)
        log(f"{loai}: lần {attempt}, số dòng train tối đa = {rows or 'toàn bộ'}")
        logf = open(f"{WORK}/log_{ten}.txt", "a")
        p = subprocess.Popen([sys.executable, "-u", script_path], env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="", flush=True); logf.write(line)
        rc = p.wait(); logf.close()
        if rc == 0 and os.path.exists(rep_path):
            if not KEEP_DATA and data.startswith("/tmp/data/"):   # chỉ xóa thư mục dữ liệu do chính notebook tạo
                shutil.rmtree(data, ignore_errors=True)
            return "OK" if rows is None else f"OK_SUBSAMPLE_{rows}"
        log(f"{loai}: lần {attempt} thất bại (mã thoát {rc})")
        txt = open(f"{WORK}/log_{ten}.txt").read()
        if "Không đủ đĩa" in txt:
            return "FAILED_DISK"          # thử lại vô ích
        if "Read-only file system" in txt or "Hệ tệp không ghi được" in txt:
            return "FAILED_FS_READONLY"   # thử lại vô ích: cần restart phiên
        if rc not in (-9, 137) and "MemoryError" not in txt:
            return "FAILED"               # chỉ thử lại (ít dòng hơn) khi nghi hết RAM
    return "FAILED"

status = {}
for loai in TYPES:
    try:
        status[loai] = run_type(loai)
    except Exception as e:                # không để một loại làm hỏng loại kia
        status[loai] = f"FAILED_DRIVER: {e!r}"
    log(f"{loai} => {status[loai]}")

# ---- Gom kết quả: model_card NHÁP + summary + zip ----
bundle_dir = f"{WORK}/bundle"; shutil.rmtree(bundle_dir, ignore_errors=True); os.makedirs(bundle_dir)
summary = {"trang_thai": status, "tong_phut": round((time.time()-T0)/60, 1), "loai": {}}
for loai in TYPES:
    ten = loai.lower().replace("_", "")
    rep_path, model = f"{WORK}/report_{ten}.json", f"{WORK}/ember2024_{ten}.txt"
    if not (os.path.exists(rep_path) and os.path.exists(model)):
        tail = ""
        lp = f"{WORK}/log_{ten}.txt"
        if os.path.exists(lp): tail = "".join(open(lp).readlines()[-25:])
        summary["loai"][loai] = {"ket_qua": "KHÔNG CÓ MÔ HÌNH", "log_cuoi": tail}
        continue
    rep = json.load(open(rep_path))
    sha = hashlib.sha256(open(model, "rb").read()).hexdigest()
    assert sha == rep["sha256_model"], "sha256 không khớp báo cáo"
    d = f"{bundle_dir}/models/{ten}"; os.makedirs(d)
    shutil.copy(model, d)
    card = {  # 4 trường đầu là bắt buộc của src/detection/ml/loader.py
        "model_version": f"ember2024-{ten}-{sha[:8]}", "feature_schema_version": "thrember-v3-2568",
        "artifact_file": os.path.basename(model), "artifact_sha256": sha,
        "status": "DRAFT_CHUA_DUYET", "loai_file": loai, "nguon_du_lieu": rep["nguon_du_lieu"],
        "danh_gia_ember2024": rep["danh_gia"], "n_train": rep["n_train"], "phien_ban": rep["phien_ban"],
        "luu_y": rep["luu_y"]}
    json.dump(card, open(f"{d}/model_card.json", "w"), indent=1, ensure_ascii=False)
    summary["loai"][loai] = {"ket_qua": status[loai], "sha256": sha, "n_train": rep["n_train"],
                             "train_phut": rep["train_phut"], "danh_gia": rep["danh_gia"], "phien_ban": rep["phien_ban"]}
json.dump(summary, open(f"{bundle_dir}/summary.json", "w"), indent=1, ensure_ascii=False)
for fn in os.listdir(WORK):
    if fn.startswith(("report_", "log_", "pip_freeze_", "lgbm_config")):
        shutil.copy(f"{WORK}/{fn}", bundle_dir)

ZIP_NAME = "dacn_ml_bundle_" + "_".join(t.lower().replace("_", "") for t in TYPES) + ".zip"
zip_path = f"{WORK}/{ZIP_NAME}"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(bundle_dir):
        for f in files:
            fp = os.path.join(root, f); z.write(fp, os.path.relpath(fp, bundle_dir))
shutil.rmtree(bundle_dir, ignore_errors=True)

print("\n" + "=" * 70)
print("XONG. TẢI 1 FILE: " + ZIP_NAME + " (tab Output). Dán khối dưới đây cho Claude:")
print("=" * 70)
print(json.dumps(summary, indent=1, ensure_ascii=False))
log(f"zip: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
